In [1]:
import os
import re
import pandas as pd
from pathlib import Path

AvanaRawReadcounts.csv:  Raw sgRNA read counts for all screens and pDNA baselines.  
AvanaGuideMap.csv:       Mapping between sgRNAs and their target genes.  
ScreenSequenceMap.csv:   Metadata for replicates: cell line, QC status, pDNA indicator.

In [2]:
# Mac Data Paths
raw   = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaRawReadcounts.csv"))
guide = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaGuideMap.csv"))
seqmap = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/ScreenSequenceMap.csv"))

In [8]:
# Linux Data Paths#
raw   = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaRawReadcounts.csv"))
guide = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaGuideMap.csv"))
seqmap = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/ScreenSequenceMap.csv"))

### Exploratory codes

In [3]:
raw.shape, guide.shape, seqmap.shape

((74687, 2285), (170429, 6), (3238, 12))

In [4]:
"""
raw.columns[-10:].tolist()
raw.columns[raw.columns.str.startswith('pDNA_batch')].tolist()
with open(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/notebooks/raw_columns.txt"), "w") as f:
    for col in raw.columns.tolist():
        f.write(col + "\n")
"""

'\nraw.columns[-10:].tolist()\nraw.columns[raw.columns.str.startswith(\'pDNA_batch\')].tolist()\nwith open(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/notebooks/raw_columns.txt"), "w") as f:\n    for col in raw.columns.tolist():\n        f.write(col + "\n")\n'

In [5]:
#seqmap['pDNABatch'].unique()
print(raw.iloc[:, -3:].head(3))
print("---------------------------------------------")
#print(seqmap[seqmap['pDNABatch'].str.startswith('Avana', na=False)].head(3))
seqmap[seqmap['pDNABatch'].str.startswith('Avana', na=False)].head(3)


   pDNA_batch_Avana-4  pDNA_batch_Avana-3  pDNA_batch_Avana-2
0              2165.0              6327.0               190.5
1              3127.5              9322.0               289.0
2               517.0              2030.5                41.5
---------------------------------------------


,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
0,Colo699-311cas9-RepA-p6_Avana-4,SC-001041.AV01,21,Avana-4,A,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
1,Colo699-311cas9-RepB-p6_Avana-4,SC-001041.AV01,21,Avana-4,B,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
2,HSB-2-311cas9_RepA_p4_Avana-3,SC-001737.AV01,21,Avana-3,A,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True


In [6]:
seqmap['Library'].unique()

array(['Avana', 'Humagne-CD', 'KY'], dtype=object)

In [22]:
print('pDNA Batch: ',seqmap['pDNABatch'].unique())
print('SCreen Type: ',seqmap['ScreenType'].unique())
print('Library: ',seqmap['Library'].unique())
print(seqmap['ExcludeFromCRISPRCombined'].unique())


pDNA Batch:  ['Avana-4' 'Avana-3' 'Avana-2' 'CD-HiSeq' 'CD-NovaSeq' 'KY-1' 'KY-2']
SCreen Type:  ['2DS' 'pDNA']
Library:  ['Avana' 'Humagne-CD' 'KY']
[False]


# Step 3

In [7]:
seqmap[seqmap['SequenceID'].isna()]

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC


In [23]:
rep_good = seqmap[seqmap['SequenceID'].notna() & 
                  (seqmap['ScreenType'] == '2DS') &
                  (seqmap['ExcludeFromCRISPRCombined'] == False) &
                  (seqmap['Library']=='Avana') & 
                  seqmap['PassesQC'] == True]

pdna_rows = seqmap[seqmap['SequenceID'].notna() & 
    (seqmap['ScreenType']== 'pDNA') &
    seqmap['pDNABatch'].notna()]

print('seqmap shape:', seqmap.shape)
print('rep_good shape:', rep_good.shape)
print(f"pdna_rows shape: {pdna_rows.shape}")


seqmap shape: (3238, 12)
rep_good shape: (2148, 12)
pdna_rows shape: (7, 12)


### Step 3.1 Understand your raw readcount matrix orientation

In [26]:
raw.rename(columns={'Guidesequence': 'sgRNAs'}, inplace=True)

Do column names in "raw" match rep_map['SequenceID']?

In [27]:
print(f"set results: {len(set(rep_good['SequenceID']) & set(raw.columns))}")
print(f"rep_good columns: {len(rep_good['SequenceID'].unique())}")
print(f"raw columns: {len(raw.columns)}")

set results: 2148
rep_good columns: 2148
raw columns: 2285


In [29]:
A = set(rep_good['SequenceID'])
B = set(raw.columns)

non_overlap = A - B
non_overlap

set()

In [ ]:
print('Number of rows in rep_good:', rep_good.shape[0])
print('Number of columns in raw:', raw.shape[1])

Number of rows in rep_good: 2148
Number of columns in raw: 2285


: 